In [ ]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
import tensorflow as tf
from transformers import TFAutoModel

In [ ]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
import pandas as pd
import json
df_psytar = pd.read_csv("data/PsyTAR.csv")
df_psytar.head(5)

In [ ]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
# for reproducing and fixing purposes, due to the cadec dataset not found
# df = pd.concat([df_psytar.iloc[:df_psytar.shape[0]+1], df_cadec])
df=df_psytar

In [ ]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
df_1 = df[df['ADR']==1]
df_0 = df[df['ADR']==0]

In [ ]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
df_0 = df_0.sample(df_1.shape[0])

In [ ]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
df = pd.concat([df_1,df_0])

In [ ]:
# --- [CELL 6]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
# --- [CELL 7]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
def process_data(row):

    text = row['sentences']
    text = str(text)
    text = ' '.join(text.split())

    encodings = tokenizer(text, padding="max_length", truncation=True, max_length=128)

    label = 0
    if row['ADR'] == 1:
        label += 1

    encodings['label'] = label
    encodings['text'] = text

    return encodings

In [ ]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
processed_data = []

for i in range(len(df[:1000])):
    processed_data.append(process_data(df.iloc[i]))

In [ ]:
# --- [CELL 9]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 10}
train_data = df["sentences"]
train_labels = df['ADR']

In [ ]:
# --- [CELL 10]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 11}
from sklearn.model_selection import train_test_split

new_df = pd.DataFrame(processed_data)

train_df, valid_df = train_test_split(
    new_df,
    test_size=0.2,
    random_state=2022
)

In [ ]:
# --- [CELL 11]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 12}
import pyarrow as pa
from datasets import Dataset

train_hg = Dataset(pa.Table.from_pandas(train_df))
valid_hg = Dataset(pa.Table.from_pandas(valid_df))

In [ ]:
# --- [CELL 12]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 13}
# === BEFORE (original) ===
# class HuggingFaceLayer(tf.keras.layers.Layer):
#     def __init__(self, model_name, output_hidden_states=False, trainable=False, **kwargs):
#         super(HuggingFaceLayer, self).__init__(**kwargs)
#         self.model = TFAutoModel.from_pretrained(model_name, output_hidden_states=output_hidden_states)
#         self.trainable = trainable
# 
#     def build(self, input_shape):
#         self.model.built = True
#         if not self.trainable:
#             self.model.trainable = False
#         super(HuggingFaceLayer, self).build(input_shape)
# 
#     def call(self, inputs, **kwargs):
#         outputs = self.model(inputs, **kwargs)
#         return outputs

# === AFTER (edited) ===
class HuggingFaceLayer(tf.keras.layers.Layer):
    def __init__(self, model_name, output_hidden_states=False, trainable=False, **kwargs):
        super(HuggingFaceLayer, self).__init__(**kwargs)
        self.model = TFAutoModel.from_pretrained(model_name, output_hidden_states=output_hidden_states)
        self.trainable = trainable

    def build(self, input_shape):
        self.model.built = True
        if not self.trainable:
            self.model.trainable = False
        super(HuggingFaceLayer, self).build(input_shape)

    def call(self, inputs, training=None):
        input_ids = tf.cast(inputs, tf.int32)
        outputs = self.model(input_ids=input_ids, training=training)
        return outputs.last_hidden_state[:, 0, :]

In [ ]:
# --- [CELL 13]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 14}
model_name = 'bert-base-uncased'
model = tf.keras.Sequential()
model.add(HuggingFaceLayer(model_name=model_name))
model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

In [ ]:
# --- [CELL 14]: ---
# cell_state: edited
# execution_status: {'status': 'timeout', 'done': True, 'execution_count': 15}
# === BEFORE (original) ===
# # Compile and train the model
# model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# model.fit(train_data, train_labels, epochs=10)

# === AFTER (edited) ===
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# tokenize raw text into integer ids compatible with embedding/index-based models
max_len = 128
vocab_size = 20000

keras_tokenizer = Tokenizer(num_words=vocab_size, oov_token='[OOV]')
keras_tokenizer.fit_on_texts(train_data.astype(str).tolist())

X_train = keras_tokenizer.texts_to_sequences(train_data.astype(str).tolist())
X_train = pad_sequences(X_train, maxlen=max_len, padding='post', truncating='post')

y_train = train_labels.values

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=10)

In [ ]:
import numpy as np

assert isinstance(model, tf.keras.Model)

try:
    input_names = sorted([tensor.name.split(':')[0] for tensor in model.inputs])
    assert input_names == ['attention_mask', 'input_ids'], input_names
except ValueError:
    # Defer input-signature validation to the prediction check below
    pass

# Verify HuggingFaceLayer (Fix 2 + implicit Fix 1)
layer_types = [type(layer).__name__ for layer in model.layers]
assert 'HuggingFaceLayer' in layer_types, \
    f"HuggingFaceLayer must be used. Found: {layer_types}"

# Explicitly validate Fix 1: CLS token extraction produces correct dimension
batch_ids = np.array(train_hg['input_ids'][:2], dtype=np.int32)
batch_mask = np.array(train_hg['attention_mask'][:2], dtype=np.int32)
hf_layer_output = HuggingFaceLayer(model_name)(inputs={'input_ids': batch_ids, 'attention_mask': batch_mask})
assert hf_layer_output.shape[-1] == 768, \
    f"CLS token should be 768-dim (BERT hidden size), got {hf_layer_output.shape[-1]}"

# Validate Fix 3: Model training with dict inputs worked
assert hasattr(model, 'history') and model.history is not None
assert 'loss' in model.history.history and len(model.history.history['loss']) >= 1

# Validate full pipeline prediction
preds = model.predict({'input_ids': batch_ids, 'attention_mask': batch_mask}, verbose=0)
assert preds.shape == (2, 1), preds.shape
assert np.isfinite(preds).all()